# RTENSO2026 El Nino cases: U.S. temperature and precipitation

Monthly versions of the CPC ENSO-update panels for the contiguous U.S., for each event in `EVENTS`
and each start:

* **Departures** (CPC "last 30/90 days"): percent of average precipitation and temperature
  departure (degC) over 1-month and 3-month windows (`DEPARTURE_WINDOWS`), for the observations and
  each case's ensemble mean.
* **Seasonal outlook** (CPC tercile outlook): probability that the season (`OUTLOOK`) is above,
  near or below normal, from the share of forecast members beyond the observed 1981-2010 tercile
  bounds. Shading shows the most likely category where its probability reaches `OUTLOOK_MIN_PROB`;
  elsewhere "equal chances" (white). The RTENSO2026 cases in `OUTLOOK_POOL` are pooled (members that
  repeat an already pooled case exactly are left out); the observed category is shown when the
  season has been observed.

Forecast anomalies remove the Reanalysis 1981-2010 lead-time climatology (lead-drift corrected for
Feb/Aug starts, `CLIM_METHOD`); forecast percent of average adds them to the observed climatology.
Observations: temperature from ERA5-Land (through Sep 2026), precipitation from GPCP v2.3 (through
Jun 2026), each against its own 1981-2010 climatology.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings

import cftime
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader

from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import BoundaryNorm, ListedColormap
from shapely import contains_xy
from shapely.ops import unary_union

import rtenso_case as rc

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="All-NaN slice")

### Run controls

In [ ]:
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"         # Feb/Aug drift correction: "lead_drift" or "target_month"
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
               9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

EVENTS = {
    2015: {"label": "2015/16 El Niño", "init_months": (2, 5, 8, 11)},
    2026: {"label": "2026/27 El Niño", "init_months": (2, 5, 8)},
}
# Windows as (years after the event year, month); CPC uses the last 30 and 90 days
DEPARTURE_WINDOWS = {"30-day (1-month)": [(0, 12)], "90-day (3-month)": [(0, 10), (0, 11), (0, 12)]}
OUTLOOK = {"label": "NDJ", "months": [(0, 11), (0, 12), (1, 1)]}   # season of the tercile outlook
OUTLOOK_MIN_PROB = 0.4             # below this the outlook shows equal chances

OBS_REFERENCE = {"TREFHT": "ERA5_Land"}   # PRECT: rc default (GPCP v2.3, archive past 2017)
FIELDS = ["PRECT", "TREFHT"]

MODEL_PREFIX = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5"
CASES = {
    "Reanalysis-CLIM": {"case_prefix": f"{MODEL_PREFIX}_RTENSO2026_clim", "init_months": (2, 5, 8, 11)},
    "Reanalysis-OBS": {"case_prefix": f"{MODEL_PREFIX}_RTENSO2026_obs", "init_months": (2, 5, 8, 11)},
    "Reanalysis-NSIDC": {"case_prefix": f"{MODEL_PREFIX}_RTENSO2026_obs_ice_nsidc", "init_months": (2, 5, 8, 11)},
    "Reanalysis-FOSIRL": {"case_prefix": f"{MODEL_PREFIX}_BruteForce", "init_months": (2, 5, 8, 11)},
}
CASES = {name: case for name, case in CASES.items() if case["init_months"]}   # an empty tuple drops a case
OUTLOOK_POOL = ("Reanalysis-CLIM", "Reanalysis-OBS", "Reanalysis-NSIDC")   # pooled into one outlook; other cases get their own row
US_BOX = (230.0, 300.0, 20.0, 55.0)   # data subset (0-360 longitudes)

### Load observations and forecasts

In [ ]:
%%time
def subset(da):
    lon0, lon1, lat0, lat1 = US_BOX
    return da.sel(lon=slice(lon0, lon1), lat=slice(lat0, lat1))


def window_times(year, months):
    return [cftime.DatetimeNoLeap(year + dy, m, 15) for dy, m in months]


def window_mean(da, times):
    """Mean over the window months; NaN unless every month is present (forecasts: by target month)."""
    if "lead" in da.dims:
        da = da.swap_dims(lead="time").drop_vars("lead")
    return da.reindex(time=times).mean("time", skipna=False)


def try_obs_anomaly(field, years):
    try:
        return subset(rc.obs_anomaly(field, years=years, mode=derivation_mode, product=OBS_REFERENCE.get(field))).load()
    except ValueError as err:
        print(f"no {field} observations for {years}: {err}")
        return None


# U.S. mask (contiguous states) on the 1x1 grid
countries = shpreader.Reader(shpreader.natural_earth(resolution="110m", category="cultural", name="admin_0_countries"))
usa = unary_union([r.geometry for r in countries.records() if r.attributes.get("ADM0_A3") == "USA"])
grid = subset(rc.model_grid())
lon2d, lat2d = np.meshgrid(grid["lon"].values - 360.0, grid["lat"].values)
us_mask = xr.DataArray(contains_xy(usa, lon2d, lat2d) & (lat2d < 50) & (lat2d > 24),
                       coords={"lat": grid["lat"], "lon": grid["lon"]}, dims=("lat", "lon"))

obs_clim = {f: subset(rc.obs_monthly_climatology(f, mode=derivation_mode, product=OBS_REFERENCE.get(f))).load() for f in FIELDS}
obs_hist = {f: try_obs_anomaly(f, rc.CLIM_YEARS) for f in FIELDS}                   # for the tercile bounds
obs_anom = {(year, f): try_obs_anomaly(f, (year, year + 2)) for year in EVENTS for f in FIELDS}

fc = {}   # (year, case, init, field) -> member anomalies (member, lead, lat, lon) with target "time"
for year, ev in EVENTS.items():
    for init in ev["init_months"]:
        for case in CASES:
            if init not in CASES[case]["init_months"]:
                continue
            anoms = {f: rc.hindcast_anomaly(CASES[case]["case_prefix"], init, f, year=year, method=CLIM_METHOD, mode=derivation_mode)
                     for f in FIELDS}
            if any(a is None for a in anoms.values()):
                print(f"[{year} {case} init{init:02d}] not available")
                continue
            for f, a in anoms.items():
                fc[(year, case, init, f)] = subset(a).load()
            print(f"[{year} {case} init{init:02d}] {anoms['PRECT'].sizes['member']} members")

### Figure setup

In [ ]:
fontz = 24        # base font size; every other size scales from it
fig_type = "png"  # saved figure type: "png", "pdf" (vector), "eps", "svg", ...
fig_dpi = 300     # resolution of saved raster figures
FIG = {
    "panel": (9.0, 5.6),            # inches per map panel (width, height)
    "hspace": 0.25,
    "wspace": 0.05,
    "title_space": 1.6,             # inches above the panels for the two-line suptitle
    "legend_space": 1.7,            # inches below the panels for the colorbars / outlook legend
    "suptitle": fontz * 1.0,
    "title": fontz * 0.7,
    "label": fontz * 0.75,
    "tick": fontz * 0.6,
    "extent": [-124.5, -66.5, 23.5, 50.0],
    "projection": ccrs.LambertConformal(central_longitude=-96.0, standard_parallels=(33.0, 45.0)),
    "t_lines": True,                # black temperature contours, as in the CPC departure maps
}
PANEL_LETTERS = "abcdefghijklmnopqrstuvwxyz"
# CPC departure colour scales
PCT_LEVELS = [10, 25, 50, 75, 125, 150, 200, 400, 800]
PCT_COLORS = ["#4a2c1f", "#7a5244", "#a67c6d", "#d7b8ae", "#ffffff", "#ddf5d6", "#a8e49a", "#5fc552", "#2f9a2b", "#3d7be0"]
T_LEVELS = np.arange(-8, 9, 1)
T_COLORS = ["#c6b9f4", "#8f73e5", "#4935c4", "#2c1f97", "#2e62d4", "#478ce8", "#7cb2f2", "#ace0f8",
            "#fff9b5", "#f6c25f", "#f0a33a", "#e86327", "#d02e1d", "#8c1a14", "#efd7cc", "#a07d6f"]
# Outlook probability bins and colours (light to dark with increasing probability)
PROB_BINS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0001]
OUTLOOK_COLORS = {
    "PRECT": {"below": ["#f2e3c0", "#dfc27d", "#c79a4e", "#a5722b", "#7e5016", "#55330a"],
              "above": ["#d5efcf", "#a6dba0", "#74c476", "#41ab5d", "#238b45", "#00441b"]},
    "TREFHT": {"below": ["#d6e5f3", "#a8c9e6", "#6fa9d6", "#3a83bf", "#145ca2", "#08306b"],
               "above": ["#fde0b5", "#fbb67d", "#f78a52", "#e65a33", "#c42b1c", "#8b0f0b"]},
    "near": ["#d9d9d9", "#a6a6a6"],
}
FIELD_TITLES = {"PRECT": "Precipitation", "TREFHT": "Temperature"}


def us_map(fig, nrow, ncol, index, title):
    ax = fig.add_subplot(nrow, ncol, index, projection=FIG["projection"])
    ax.set_extent(FIG["extent"], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor="0.25", zorder=4)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=4)
    ax.add_feature(cfeature.BORDERS, linewidth=0.7, zorder=4)
    ax.set_title(title, fontsize=FIG["title"], loc="left")
    return ax


def empty_panel(ax, text):
    ax.text(0.5, 0.5, text, transform=ax.transAxes, ha="center", va="center", fontsize=FIG["label"], color="0.4",
            bbox={"facecolor": "white", "edgecolor": "0.6", "pad": 6}, zorder=6)


def set_margins(fig):
    """Fixed title and legend space in inches, whatever the number of rows."""
    height = fig.get_figheight()
    fig.subplots_adjust(hspace=FIG["hspace"], wspace=FIG["wspace"], top=1 - FIG["title_space"] / height,
                        bottom=FIG["legend_space"] / height)
    return height


def save_fig(fig, *parts):
    print(rc.save_figure(fig, *parts, dpi=fig_dpi, ext=fig_type))

### Temperature and precipitation departures

One figure per event, start and window: rows are the observations and each case's ensemble mean, columns percent of average precipitation and temperature departure.

In [ ]:
def percent_of_average(anom, times):
    clim = window_mean(xr.concat([obs_clim["PRECT"].sel(month=t.month).drop_vars("month") for t in times], "time")
                       .assign_coords(time=times), times)
    return 100.0 * (clim + anom) / clim.where(clim > 0.05)


def draw_departure(ax, field, da):
    da = da.where(us_mask)
    tr = ccrs.PlateCarree()
    if field == "PRECT":
        levels = [0.0] + PCT_LEVELS + [1e6]
        cmap = ListedColormap(PCT_COLORS)
        mesh = ax.pcolormesh(da["lon"], da["lat"], da, cmap=cmap, norm=BoundaryNorm(levels, cmap.N), transform=tr,
                             shading="nearest", zorder=1)
        return mesh
    cmap = ListedColormap(T_COLORS[1:-1])
    cmap.set_under(T_COLORS[0]); cmap.set_over(T_COLORS[-1])
    cf = ax.contourf(da["lon"], da["lat"], da, levels=T_LEVELS[1:-1], cmap=cmap, extend="both", transform=tr, zorder=1)
    if FIG["t_lines"]:
        cs = ax.contour(da["lon"], da["lat"], da, levels=T_LEVELS, colors="k", linewidths=0.7, transform=tr, zorder=2)
        ax.clabel(cs, fmt="%g", fontsize=FIG["tick"] * 0.8, inline=True)
    return cf


def add_departure_colorbars(fig, mappables):
    for col, (field, m) in enumerate(mappables.items()):
        cax = fig.add_axes([0.13 + 0.42 * col, 0.85 / fig.get_figheight(), 0.32, 0.18 / fig.get_figheight()])
        if field == "PRECT":
            cb = fig.colorbar(m, cax=cax, orientation="horizontal", ticks=PCT_LEVELS)
            cb.set_label("Percent of average precipitation (%)", fontsize=FIG["label"])
        else:
            cb = fig.colorbar(m, cax=cax, orientation="horizontal", ticks=T_LEVELS[1:-1])
            cb.set_label("Temperature departure (°C)", fontsize=FIG["label"])
        cb.ax.tick_params(labelsize=FIG["tick"])


for year, ev in EVENTS.items():
    for init in ev["init_months"]:
        cases = [c for c in CASES if (year, c, init, "PRECT") in fc]
        if not cases:
            continue
        for wlabel, months in DEPARTURE_WINDOWS.items():
            times = window_times(year, months)
            end = times[-1]
            rows = ["Observed"] + cases
            fig = plt.figure(figsize=(FIG["panel"][0] * 2, FIG["panel"][1] * len(rows)))
            set_margins(fig)
            mappables, k = {}, 0
            for r, name in enumerate(rows):
                for c, field in enumerate(FIELDS):
                    if name == "Observed":
                        src = obs_anom[(year, field)]
                        anom = window_mean(src, times) if src is not None else None
                        title = f"({PANEL_LETTERS[k]}) Observed ({'ERA5-Land' if field == 'TREFHT' else 'GPCP v2.3'})"
                    else:
                        anom = window_mean(fc[(year, name, init, field)].mean("member"), times)
                        lead = [i for i, t in enumerate(fc[(year, name, init, field)]["time"].values) if (t.year, t.month) == (end.year, end.month)]
                        title = f"({PANEL_LETTERS[k]}) {name}, lead {lead[0] if lead else '?'}"
                    ax = us_map(fig, len(rows), 2, r * 2 + c + 1, f"{title}: {FIELD_TITLES[field].lower()}")
                    k += 1
                    if anom is None or not bool(anom.where(us_mask).notnull().any()):
                        empty_panel(ax, "Not yet observed")
                        continue
                    da = percent_of_average(anom, times) if field == "PRECT" else anom
                    mappables[field] = draw_departure(ax, field, da)
            add_departure_colorbars(fig, mappables)
            first = times[0]
            span = f"{MONTH_NAMES[end.month]} {end.year}" if len(times) == 1 else \
                f"{MONTH_NAMES[first.month]} {first.year}-{MONTH_NAMES[end.month]} {end.year}"
            fig.suptitle(f"U.S. precipitation and temperature departures, {wlabel} window {span}\n"
                         f"1 {MONTH_NAMES[init]} {year} start, {ev['label']}", fontsize=FIG["suptitle"], y=1 - 0.25 / fig.get_figheight())
            save_fig(fig, "rtenso2026", "us_departures", f"y{year}", f"init{init:02d}", f"{len(times)}mon")
            plt.show()

### Seasonal outlook (tercile probabilities)

One figure per event and start: rows are the pooled RTENSO2026 ensemble, any other case, and the observed category once the season is observed; columns precipitation and temperature.

In [ ]:
def season_series(da, months, years):
    """Window means for every year whose window lies inside ``years``, along "year"."""
    last_dy = max(dy for dy, _ in months)
    ys = [y for y in range(years[0], years[1] + 1) if y + last_dy <= years[1]]
    return xr.concat([window_mean(da, window_times(y, months)) for y in ys], dim="year").assign_coords(year=ys)


terciles = {}
for f in FIELDS:
    hist = season_series(obs_hist[f], OUTLOOK["months"], rc.CLIM_YEARS)
    terciles[f] = hist.quantile([1 / 3, 2 / 3], dim="year")


def outlook_code(p_below, p_near, p_above):
    """Category code: 0 equal chances; 1-6 below, 7-8 near, 9-14 above (by probability bin)."""
    probs = np.stack([p_below, p_near, p_above])
    top = np.nanmax(probs, axis=0)
    winner = np.nanargmax(np.where(np.isnan(probs), -1, probs), axis=0)
    unique = (probs == top).sum(axis=0) == 1
    bins = np.clip(np.digitize(top, PROB_BINS) - 1, 0, len(PROB_BINS) - 2)
    code = np.where(winner == 0, 1 + bins, np.where(winner == 1, 7 + np.minimum(bins, 1), 9 + bins))
    return np.where((top >= OUTLOOK_MIN_PROB) & unique, code, 0).astype(float)


def outlook_cmap(field):
    colors = ["white"] + OUTLOOK_COLORS[field]["below"] + OUTLOOK_COLORS["near"] + OUTLOOK_COLORS[field]["above"]
    return ListedColormap(colors), BoundaryNorm(np.arange(-0.5, len(colors)), len(colors))


def draw_outlook(ax, field, code):
    cmap, norm = outlook_cmap(field)
    code = code.where(us_mask)
    return ax.pcolormesh(code["lon"], code["lat"], code, cmap=cmap, norm=norm, transform=ccrs.PlateCarree(),
                         shading="nearest", zorder=1)


def add_outlook_legend(fig):
    """Below / Near / Above probability bars under each column, as in the CPC outlook legend."""
    labels = [f"{int(b * 100)}" for b in PROB_BINS[:-1]]
    bar, gap = 0.105, 0.025                       # figure fractions: width of a 6-bin bar and the gap between bars
    for col, field in enumerate(FIELDS):
        left = fig.axes[col].get_position().x0
        x = left
        for cat, colors in [("Below", OUTLOOK_COLORS[field]["below"]), ("Near", OUTLOOK_COLORS["near"]),
                            ("Above", OUTLOOK_COLORS[field]["above"])]:
            width = bar * len(colors) / 6
            cax = fig.add_axes([x, 0.95 / fig.get_figheight(), width, 0.16 / fig.get_figheight()])
            cb = ColorbarBase(cax, cmap=ListedColormap(colors), norm=BoundaryNorm(np.arange(len(colors) + 1), len(colors)),
                              orientation="horizontal", ticks=np.arange(len(colors) + 1))
            cb.ax.set_xticklabels(labels[:len(colors)] + ["100"], fontsize=FIG["tick"] * 0.75)
            cb.ax.set_title(f"{cat} normal", fontsize=FIG["tick"])
            x += width + gap
        fig.text(left, 0.15 / fig.get_figheight(), f"{FIELD_TITLES[field]}: probability of the most likely category (%)",
                 fontsize=FIG["tick"], ha="left")


for year, ev in EVENTS.items():
    times = window_times(year, OUTLOOK["months"])
    for init in ev["init_months"]:
        rows = {}
        pooled = {f: [] for f in FIELDS}
        for case in CASES:
            if (year, case, init, "PRECT") not in fc:
                continue
            members = {f: window_mean(fc[(year, case, init, f)], times) for f in FIELDS}
            if case in OUTLOOK_POOL:
                if any(np.array_equal(members["PRECT"].values, p.values) for p in pooled["PRECT"]):
                    print(f"[{year} init{init:02d}] {case} repeats a pooled case member for member; not pooled again")
                    continue
                for f in FIELDS:
                    pooled[f].append(members[f])
            else:
                rows[case] = members
        if pooled["PRECT"]:
            rows = {"RTENSO2026 pooled": {f: xr.concat(pooled[f], "member") for f in FIELDS}, **rows}
        if not rows:
            continue
        observed = {f: window_mean(obs_anom[(year, f)], times) if obs_anom[(year, f)] is not None else None for f in FIELDS}
        has_obs = all(o is not None and bool(o.where(us_mask).notnull().any()) for o in observed.values())
        names = list(rows) + ["Observed category"]
        fig = plt.figure(figsize=(FIG["panel"][0] * 2, FIG["panel"][1] * len(names)))
        set_margins(fig)
        k = 0
        for r, name in enumerate(names):
            for c, f in enumerate(FIELDS):
                lo, hi = terciles[f].sel(quantile=1 / 3, drop=True), terciles[f].sel(quantile=2 / 3, drop=True)
                if name == "Observed category":
                    ax = us_map(fig, len(names), 2, r * 2 + c + 1, f"({PANEL_LETTERS[k]}) Observed category: {FIELD_TITLES[f].lower()}")
                    k += 1
                    if not has_obs:
                        empty_panel(ax, "Not yet observed")
                        continue
                    o = observed[f]
                    p = [(o < lo).astype(float), ((o >= lo) & (o <= hi)).astype(float), (o > hi).astype(float)]
                else:
                    m = rows[name][f]
                    n = m.sizes["member"]
                    p = [(m < lo).sum("member") / n, ((m >= lo) & (m <= hi)).sum("member") / n, (m > hi).sum("member") / n]
                    ax = us_map(fig, len(names), 2, r * 2 + c + 1, f"({PANEL_LETTERS[k]}) {name} ({n} members): {FIELD_TITLES[f].lower()}")
                    k += 1
                code = xr.DataArray(outlook_code(*[x.values for x in p]), coords=lo.coords, dims=lo.dims)
                draw_outlook(ax, f, code)
        add_outlook_legend(fig)
        first, end = times[0], times[-1]
        fig.suptitle(f"U.S. seasonal outlook, {OUTLOOK['label']} {first.year}/{str(end.year)[2:]} "
                     f"(terciles of the observed 1981-2010 season; white: equal chances)\n"
                     f"1 {MONTH_NAMES[init]} {year} start, {ev['label']}", fontsize=FIG["suptitle"], y=1 - 0.25 / fig.get_figheight())
        save_fig(fig, "rtenso2026", "us_outlook", f"y{year}", f"init{init:02d}", OUTLOOK["label"])
        plt.show()